# DDIM Sampling

**Companion wiki page:** https://ml-viz.vercel.app/wiki/ddim-sampling

A 2-D toy diffusion model trained on a two-moons dataset, then sampled two ways: full 1000-step DDPM vs 50-step deterministic DDIM. Same model, ~20× fewer steps.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## Toy data and the noise schedule

We diffuse 2-D points instead of images — the math is identical and everything runs in seconds. Linear β schedule, with the usual $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_s \alpha_s$.

In [ ]:
def two_moons(n):
    t = np.random.rand(n) * np.pi
    x1 = np.c_[np.cos(t), np.sin(t)] + np.random.randn(n, 2) * 0.05
    x2 = np.c_[1 - np.cos(t), 0.5 - np.sin(t)] + np.random.randn(n, 2) * 0.05
    return np.vstack([x1, x2]) - [0.5, 0.25]

data = two_moons(2000)

T = 1000
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = np.cumprod(alphas)

plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0], data[:, 1], s=4, color='#2dd4bf', alpha=0.5)
plt.title('training data'); plt.axis('equal'); plt.show()

## A tiny noise-prediction network

A 2-hidden-layer MLP trained with the standard DDPM objective: sample $t$, noise the data to $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon$, predict $\boldsymbol\epsilon$. (NumPy forward/backward — no framework needed at this scale.)

In [ ]:
H = 128
rng = np.random.default_rng(1)
params = {
    "W1": rng.normal(scale=0.3, size=(3, H)), "b1": np.zeros(H),
    "W2": rng.normal(scale=0.3, size=(H, H)), "b2": np.zeros(H),
    "W3": rng.normal(scale=0.3, size=(H, 2)), "b3": np.zeros(2),
}

def net(x, t):
    # x: (n, 2); t: (n,) in [0, 1]
    inp = np.c_[x, t]
    h1 = np.maximum(0, inp @ params["W1"] + params["b1"])
    h2 = np.maximum(0, h1 @ params["W2"] + params["b2"])
    return h2 @ params["W3"] + params["b3"], (inp, h1, h2)

def train_step(x0, lr=1e-3):
    n = len(x0)
    t = np.random.randint(0, T, n)
    eps = np.random.randn(n, 2)
    xt = np.sqrt(abar[t])[:, None] * x0 + np.sqrt(1 - abar[t])[:, None] * eps
    pred, (inp, h1, h2) = net(xt, t / T)
    diff = pred - eps
    loss = (diff ** 2).mean()
    # backprop
    g3 = 2 * diff / (n * 2)
    params["W3"] -= lr * h2.T @ g3; params["b3"] -= lr * g3.sum(0)
    g2 = (g3 @ params["W3"].T) * (h2 > 0)
    params["W2"] -= lr * h1.T @ g2; params["b2"] -= lr * g2.sum(0)
    g1 = (g2 @ params["W2"].T) * (h1 > 0)
    params["W1"] -= lr * inp.T @ g1; params["b1"] -= lr * g1.sum(0)
    return loss

losses = [train_step(data[np.random.choice(len(data), 256)]) for _ in range(4000)]
plt.figure(figsize=(8, 3))
plt.plot(np.convolve(losses, np.ones(50) / 50, mode='valid'), color='#6366f1')
plt.title('training loss (smoothed)'); plt.grid(alpha=0.3); plt.show()

## DDPM sampling: all 1000 steps, stochastic

The baseline: walk back one noise level at a time, injecting fresh noise at every step.

In [ ]:
def sample_ddpm(n):
    x = np.random.randn(n, 2)
    for t in range(T - 1, -1, -1):
        eps_pred, _ = net(x, np.full(n, t / T))
        coef = betas[t] / np.sqrt(1 - abar[t])
        mean = (x - coef * eps_pred) / np.sqrt(alphas[t])
        x = mean + (np.sqrt(betas[t]) * np.random.randn(n, 2) if t > 0 else 0)
    return x

ddpm_samples = sample_ddpm(1000)
print("done: 1000 network calls per sample batch")

## DDIM sampling: 50 steps, deterministic

The procedure from the wiki page:

1. choose a timestep subsequence,
2. predict noise,
3. estimate $\hat{\mathbf{x}}_0 = \dfrac{\mathbf{x}_{\tau} - \sqrt{1-\bar\alpha_\tau}\,\hat\epsilon}{\sqrt{\bar\alpha_\tau}}$,
4. re-noise the estimate to the next visited level with the *same* predicted noise.

In [ ]:
def sample_ddim(n, S=50):
    taus = np.linspace(T - 1, 0, S).astype(int)
    x = np.random.randn(n, 2)
    for s in range(S - 1):
        t, t_next = taus[s], taus[s + 1]
        eps_pred, _ = net(x, np.full(n, t / T))
        x0_hat = (x - np.sqrt(1 - abar[t]) * eps_pred) / np.sqrt(abar[t])
        x = np.sqrt(abar[t_next]) * x0_hat + np.sqrt(1 - abar[t_next]) * eps_pred
    return x

ddim_samples = sample_ddim(1000, S=50)
print("done: 50 network calls — 20× fewer")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, pts, title in [
    (axes[0], data, 'training data'),
    (axes[1], ddpm_samples, 'DDPM (1000 steps)'),
    (axes[2], ddim_samples, 'DDIM (50 steps)'),
]:
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5,
               color='#2dd4bf' if title == 'training data' else '#6366f1')
    ax.set_title(title); ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.2, 1.2)
plt.tight_layout(); plt.show()

Both recover the two moons — DDIM with 5% of the compute. Because DDIM is deterministic, the same starting noise always maps to the same sample, which is what enables semantic interpolation in image models.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — the clean-image estimate

**Recap:** the heart of each DDIM step is recovering $\hat{\mathbf{x}}_0$ from the current noisy point and the predicted noise:

$$\hat{\mathbf{x}}_0 = \frac{\mathbf{x}_t - \sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}}$$

Implement it, then verify on a synthetic case where we *know* the answer: if we noise a known $\mathbf{x}_0$ with a known $\epsilon$, the formula must give $\mathbf{x}_0$ back exactly.

In [ ]:
def estimate_x0(xt, eps, t):
    # TODO(you): implement the formula using abar[t].
    ...

# known ground truth
x0_known = np.array([[0.3, -0.7]])
eps_known = np.array([[1.1, 0.4]])
t_test = 600
xt_test = np.sqrt(abar[t_test]) * x0_known + np.sqrt(1 - abar[t_test]) * eps_known

x0_rec = estimate_x0(xt_test, eps_known, t_test)
x0_rec

In [ ]:
# Run me — passes silently when correct
assert x0_rec is not None and not isinstance(x0_rec, type(Ellipsis)), "fill in the TODO first"
assert np.allclose(np.asarray(x0_rec), x0_known, atol=1e-10), "must invert the noising exactly"
print()

<details>
<summary>Solution</summary>

```python
def estimate_x0(xt, eps, t):
    return (xt - np.sqrt(1 - abar[t]) * eps) / np.sqrt(abar[t])
```
</details>